# 05b. Оси голов: reshape и transpose

Лекция: [Оси multi-head attention](../../notes/interview-prep/08e-head-axes.md).

Это подготовка к векторизованному attention, не полный новый слой. Сегодня только A. Сначала разберём числа, затем форму (B,H,T,d_head). Предыдущие ответы и код не нужно переносить.

In [1]:
import torch

B, T, D, H = 1, 3, 4, 2
head_dim = D // H
# Представим, что это УЖЕ результат общей Query-проекции, а не исходный X.
q_flat = torch.arange(B * T * D).reshape(B, T, D)
print(q_flat)
print('Форма:', q_flat.shape)

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]]])
Форма: torch.Size([1, 3, 4])


## A. Разделяем последнюю ось

Для первой позиции вектор [0,1,2,3] должен превратиться в две группы: [0,1] и [2,3]. reshape меняет форму с сохранением числа элементов; здесь порядок чисел не меняется.

Нужна форма (B,T,H,head_dim). Заполни аргументы reshape. Каждая голова в реальном attention получает признаки из обучаемой проекции полного X; не режем исходный X на независимые половины.

In [2]:
q_split = q_flat.reshape(B, T, H, head_dim)
assert isinstance(q_split, torch.Tensor)
assert q_split.shape == (B, T, H, head_dim)
assert torch.equal(q_split[0, 0, 0], torch.tensor([0, 1]))
assert torch.equal(q_split[0, 0, 1], torch.tensor([2, 3]))
assert torch.equal(q_split[0, 1, 0], torch.tensor([4, 5]))
print(q_split)
print('Форма:', q_split.shape)

tensor([[[[ 0,  1],
          [ 2,  3]],

         [[ 4,  5],
          [ 6,  7]],

         [[ 8,  9],
          [10, 11]]]])
Форма: torch.Size([1, 3, 2, 2])


### Остановка A

1. Что означает каждая ось (B,T,H,head_dim)?
2. Какие числа находятся в q_split[0,2,1]?
3. Меняет ли reshape значения параметров или обучает модель?

**Мои ответы:**

1. B - батчи,T -кол-во токенов в последовательности,H - количество голов,head_dim- выходной вектор одной головы
2. [ 8,  9]
3. нет, нет. Просто меняет формат записи, добавляет оси

**Разбор ментора (исходные ответы сохранены):**

Код reshape(B,T,H,head_dim) правильный; проверки прошли по сохранённым outputs.
1. B — количество последовательностей в одном batch, а не количество батчей. T — число позиций в последовательности. H — количество голов. head_dim — число координат Query-вектора одной головы (здесь мы группируем Q; аналогично будет для K/V).
2. Правильный ответ [10,11]. Индексы [0,2,1] означают первую последовательность, третью позицию, вторую голову. Сначала q_split[0,2] даёт [[8,9],[10,11]], затем индекс головы 1 выбирает вторую строку. [8,9] соответствует q_split[0,2,0]. Последняя ось не указана, поэтому берём весь вектор головы.
3. Верно: reshape не обучает модель и не меняет значения параметров. Уточнение: он меняет форму при сохранении числа элементов; не обязательно добавляет оси, может также объединять их. Здесь ось D=4 разбита на H=2 и head_dim=2.

Перед частью B закрепим чтение индексов: назови числа в q_split[0,1,1] и объясни выбор позиции и головы.

## B. Переставляем позиции и головы

Хотим (B,H,T,head_dim), чтобы у каждой головы была своя матрица (T,head_dim). transpose(a,b) меняет местами две оси. Не создаёт новые обучаемые параметры.

In [3]:
q_split.shape

torch.Size([1, 3, 2, 2])

In [5]:
q_heads = q_split.transpose(2,1)
assert isinstance(q_heads, torch.Tensor)
assert q_heads.shape == (B, H, T, head_dim)
assert torch.equal(q_heads[0, 0], torch.tensor([[0,1],[4,5],[8,9]]))
assert torch.equal(q_heads[0, 1], torch.tensor([[2,3],[6,7],[10,11]]))
print('Голова 0:', q_heads[0, 0])
print('Голова 1:', q_heads[0, 1])

# Одинаковая форма не гарантирует правильное расположение элементов!
wrong = q_flat.reshape(B, H, T, head_dim)
assert wrong.shape == q_heads.shape
assert not torch.equal(wrong, q_heads)
print('Неправильное прямое reshape:', wrong)

Голова 0: tensor([[0, 1],
        [4, 5],
        [8, 9]])
Голова 1: tensor([[ 2,  3],
        [ 6,  7],
        [10, 11]])
Неправильное прямое reshape: tensor([[[[ 0,  1],
          [ 2,  3],
          [ 4,  5]],

         [[ 6,  7],
          [ 8,  9],
          [10, 11]]]])


### Остановка B

1. Почему прямой reshape(B,H,T,head_dim) не заменяет reshape + transpose?
2. Какая ось последняя у q_heads? Что означает q_heads[0,1,2]?

**Мои ответы:**

1. Потому что тогда у нас будут перемешены головы и токены.
2. Последняя ось - это размерность эмбединга конкретной головы конкретного токена. Это значит первая последовательность, вторая голова, третий токен

**Разбор ментора (исходные ответы сохранены):**

Код q_split.transpose(2,1) правильный: он меняет местами H и T и эквивалентен transpose(1,2). Проверки значений прошли по сохранённым outputs.
1. Верно. Прямой reshape группирует числа в их текущем порядке, но не выполняет нужную перестановку осей. В нашем примере он ошибочно относит [2,3] к голове 0 позиции 1, хотя это Query головы 1 позиции 0. Поэтому правильная форма сама по себе не доказывает правильное соответствие токенов и голов.
2. Индексы прочитаны верно. Точнее, последняя ось — координаты Query-вектора одной головы, а не исходного token embedding. q_heads[0,1,2] выбирает весь Query-вектор второй головы третьей позиции первой последовательности: [10,11].

Часть B проверена. Далее обратная сборка в части C: вернуть (B,T,H,head_dim), затем объединить последние две оси в D.

## C. Возвращаемся к объединённым признакам

После attention выход каждой головы будет иметь такой же порядок осей (B,H,T,head_dim). Чтобы склеить признаки одной позиции, сначала вернём T перед H, а потом объединим H и head_dim через reshape.

Для упражнения используем q_heads. После transpose память может быть non-contiguous: reshape умеет при необходимости создать копию, а view не всегда допустим. Здесь используем reshape.

In [6]:
restored = q_heads.transpose(1,2).reshape(B, T, D) 
assert isinstance(restored, torch.Tensor)
assert restored.shape == q_flat.shape
assert torch.equal(restored, q_flat)
print('Обратная сборка: OK')

Обратная сборка: OK


### Остановка C

1. Почему перед объединением H и head_dim нужно вернуть порядок (B,T,H,head_dim)?
2. Какие проверки обнаруживают ошибку, которую не заметит проверка shape?

**Мои ответы:**

1. Потому что объединять мы будем головы между собой, а не проекции токенов
2. Проверки самого значения тензора

**Разбор ментора (исходные ответы сохранены):**

Код q_heads.transpose(1,2).reshape(B,T,D) правильный; по сохранённому output исходный тензор восстановлен точно.
1. По смыслу верно. Точнее: нужно объединить признаки всех голов для одной и той же позиции. В порядке (B,T,H,head_dim) последние две оси H и head_dim относятся к фиксированным B/T и объединяются в D. Прямой reshape из (B,H,T,head_dim) без перестановки может собрать в один вектор числа разных позиций. Здесь восстанавливаем учебный Q; после настоящего attention так же будем собирать контекстные векторы голов.
2. Верно. torch.equal(restored,q_flat) проверяет и форму, и точное совпадение всех элементов. Дополнительные проверки известных срезов головы/позиции помогают локализовать неправильную группировку. Для результатов вычислений с плавающей точкой, где возможны различия округления, используют torch.testing.assert_close с допуском.

Части A/B/C проверены по сохранённым результатам. Подготовка по осям завершена; полный векторизованный attention ещё не реализован.

После всех частей: Restart Kernel → Run All → сохранить outputs. Следующим шагом применим эти преобразования к Q/K/V и сравним векторизованный attention с отдельными головами на одинаковых весах.